# Topic 04 — vLLM OpenAI-compatible service

This is a bounded startup/API smoke lab, **not** the judge or a load test. Use an otherwise idle CUDA GPU. The 0.5B model, 2048-token maximum context, four-sequence cap and 45% GPU-memory utilization limit deliberately constrain the experiment. Stop the server before another GPU exercise. A free Colab GPU is not guaranteed.

Set `LSM_MODEL_CACHE` to persistent storage to reuse the immutable model snapshot; the cache is never committed. If vLLM cannot install on your runtime, report the environment as blocked rather than claiming an API run.

In [ ]:
%pip install -q 'vllm==0.29.0'


In [ ]:
import json, os, platform, shutil, socket, subprocess, sys, time
from pathlib import Path
import httpx, openai, torch, vllm
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'The service lab requires an idle CUDA GPU'
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
SERVED_NAME = 'topic04'
CACHE = Path(os.environ.get('LSM_MODEL_CACHE', str(Path.cwd() / '.cache' / 'models'))).expanduser().resolve()
CACHE.mkdir(parents=True, exist_ok=True)
try:
    MODEL_DIR = Path(snapshot_download(repo_id=MODEL_ID, revision=REVISION, cache_dir=str(CACHE), local_files_only=True))
    print('Reusing complete pinned snapshot')
except FileNotFoundError:
    print('One-time download of pinned snapshot')
    MODEL_DIR = Path(snapshot_download(repo_id=MODEL_ID, revision=REVISION, cache_dir=str(CACHE)))
assert (MODEL_DIR / 'config.json').exists() and list(MODEL_DIR.glob('*.safetensors')) and (MODEL_DIR / 'tokenizer.json').exists()
RUN = {'contract':'topic-04-api-smoke-v1', 'utc_started':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()), 'model_id':MODEL_ID, 'revision':REVISION, 'snapshot':str(MODEL_DIR), 'served_name':SERVED_NAME, 'gpu':torch.cuda.get_device_name(0), 'torch':torch.__version__, 'cuda':torch.version.cuda, 'vllm':vllm.__version__, 'openai_sdk':openai.__version__, 'python':platform.python_version()}


## Start one local server

Use `127.0.0.1`: this teaching process is not exposed to the network. The lab process is separate from the notebook kernel. Server stderr is kept only locally for incident analysis; inspect and sanitize it before sharing. Startup is allowed 5 minutes, then fails closed. The launch uses the local snapshot, so a second run does not fetch weights again.

In [ ]:
BASE_URL = 'http://127.0.0.1:8000/v1'
evidence = Path('evidence'); evidence.mkdir(exist_ok=True)
try:
    with socket.create_connection(('127.0.0.1', 8000), timeout=1): pass
except OSError: pass
else: raise RuntimeError('Port 8000 is already in use; stop the other service before this lab')
VLLM_BIN = shutil.which('vllm')
if not VLLM_BIN: raise RuntimeError('vllm CLI was not installed')
LAUNCH = [VLLM_BIN, 'serve', str(MODEL_DIR), '--served-model-name', SERVED_NAME, '--host', '127.0.0.1', '--port', '8000', '--dtype', 'float16', '--max-model-len', '2048', '--max-num-seqs', '4', '--gpu-memory-utilization', '0.45', '--enforce-eager']
RUN['launch_args'] = LAUNCH[1:]
log_file = (evidence / 'topic-04-server.local.log').open('w', encoding='utf-8')
server = subprocess.Popen(LAUNCH, stdout=log_file, stderr=subprocess.STDOUT)
def stop_server():
    if server.poll() is None:
        server.terminate()
        try: server.wait(timeout=20)
        except subprocess.TimeoutExpired: server.kill(); server.wait(timeout=10)
    if not log_file.closed: log_file.close()
__import__('atexit').register(stop_server)
deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if server.poll() is not None: raise RuntimeError(f'vLLM exited during startup: {server.returncode}; inspect local log')
    try:
        response = httpx.get(BASE_URL + '/models', timeout=2)
        if response.status_code == 200: break
    except httpx.RequestError: pass
    time.sleep(2)
else:
    server.terminate(); raise TimeoutError('vLLM did not become ready in 300 s; inspect local log')
print('Ready:', BASE_URL)


In [ ]:
client = openai.OpenAI(base_url=BASE_URL, api_key='EMPTY', timeout=30, max_retries=0)
models = client.models.list()
advertised = [item.id for item in models.data]
assert SERVED_NAME in advertised, f'served-name mismatch: {advertised}'
RUN['advertised_models'] = advertised
prompt = 'Reply with one short sentence explaining what an API server does.'
t0 = time.perf_counter()
answer = client.chat.completions.create(model=SERVED_NAME, messages=[{'role':'user','content':prompt}], max_tokens=48, temperature=0)
RUN['nonstream'] = {'wall_s':time.perf_counter()-t0, 'finish_reason':answer.choices[0].finish_reason, 'usage':answer.usage.model_dump() if answer.usage else None, 'content':answer.choices[0].message.content}
assert RUN['nonstream']['content']
RUN['nonstream']


## Streaming boundary

The SDK exposes parsed chunks; it does not expose raw SSE framing or `[DONE]`. First-content time is client-observed. The separate raw HTTP probe validates framing and terminal completion; a partial stream must never be counted as a completed response.

In [ ]:
t0 = time.perf_counter(); first_content = None; pieces = []; finish = None
stream = client.chat.completions.create(model=SERVED_NAME, messages=[{'role':'user','content':prompt}], max_tokens=48, temperature=0, stream=True)
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_content is None: first_content = time.perf_counter() - t0
        pieces.append(delta)
    if chunk.choices and chunk.choices[0].finish_reason: finish = chunk.choices[0].finish_reason
RUN['sdk_stream'] = {'first_content_s':first_content, 'wall_s':time.perf_counter()-t0, 'finish_reason':finish, 'content':''.join(pieces)}
assert first_content is not None and finish is not None and RUN['sdk_stream']['content']
payload = {'model':SERVED_NAME,'messages':[{'role':'user','content':'Say hello briefly.'}],'max_tokens':16,'temperature':0,'stream':True}
frames = []; saw_done = False
with httpx.stream('POST', BASE_URL + '/chat/completions', json=payload, timeout=30) as response:
    response.raise_for_status()
    for line in response.iter_lines():
        if line.startswith('data: '):
            body = line[6:]
            if body == '[DONE]': saw_done = True; break
            frames.append(json.loads(body))
RUN['raw_sse'] = {'json_frames':len(frames), 'done':saw_done}
assert frames and saw_done, 'partial stream or missing [DONE]'
RUN['sdk_stream'], RUN['raw_sse']


## Negative tests and token budget

A wrong served name and a rendered prompt beyond the 2048-token context must fail. Exact tokenization, not character length, drives the latter. These tests are expected errors, not a failed lab. Do not send an arbitrarily large payload: stop after crossing the bound.

In [ ]:
negative = {}
try:
    client.chat.completions.create(model='not-the-served-model', messages=[{'role':'user','content':'hi'}], max_tokens=1)
    negative['wrong_model'] = 'unexpected success'
except openai.APIStatusError as exc:
    negative['wrong_model'] = {'status':exc.status_code}
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
long_text = ' blue' * 2300
rendered = tokenizer.apply_chat_template([{'role':'user','content':long_text}], tokenize=True, add_generation_prompt=True)
assert len(rendered) > 2048
try:
    client.chat.completions.create(model=SERVED_NAME, messages=[{'role':'user','content':long_text}], max_tokens=16)
    negative['over_context'] = 'unexpected success'
except openai.APIStatusError as exc:
    negative['over_context'] = {'status':exc.status_code, 'rendered_prompt_tokens':len(rendered)}
assert isinstance(negative.get('wrong_model'), dict) and isinstance(negative.get('over_context'), dict)
RUN['negative'] = negative
negative


In [ ]:
target = evidence / 'topic-04-service.json'
target.write_text(json.dumps(RUN, indent=2, default=str), encoding='utf-8')
print('Saved:', target)
stop_server()
print('Teaching server stopped; do not leave a GPU process running.')


## Analysis

Draw the boundary from SDK → HTTP/SSE → chat template/tokenizer → engine → worker → GPU and back. Explain why `/v1/models` catches a model-name mismatch, why client-observed first content is not GPU decode time, and why a 45% memory target is not a four-request throughput guarantee. In `evidence/topic-04-analysis.md`, include one actual failure/mitigation or an explicit negative-test result. The lab did not exercise reverse proxies, concurrency or the judge's 3B model.